In [ ]:
import torch
import numpy as np
from Network import VAE, VAE_41
from torch.utils.data import DataLoader
from Util.data_loader import loader_cre
from Util.metrics import SSIM
from Util.train_41utils import train_vae_41, plot_curve
from Util.Visualize import Plot_2
import os
import torch.optim as optim


In [ ]:
import numpy as np
import random

file_dir = "../v41_data/modulus_data_2.npy"
data_np = np.load(file_dir)
data_np = torch.from_numpy(data_np).float()
#indices_to_remove = np.arange(12, 1000, 11) 
#data_np = np.delete(data_np, indices_to_remove, axis=0)
print(data_np.shape)


#print(f"全局最大应变值为: {global_max.item()}")
#print(f"全局最小应变值为: {global_min.item()}")


(1001, 41, 41)


In [ ]:
# 2. 直接进行缩放，得到归一化后的数据
# 得到的结果范围在 [-1, 1] 之间
global_max = data_np.max()
global_min = data_np.min()
data_normalized = (data_np - global_min) /(global_max - global_min)

# # 3. 增加通道维度以符合卷积输入 (Batch, Channel, H, W) -> (1000, 1, 41, 41)
data_input = data_normalized.unsqueeze(1)

In [ ]:
type(data_normalized[1])

In [ ]:
import matplotlib.pyplot as plt
random_indices = random.sample(range(data_normalized.shape[0]), 5)

# 创建一个1x10的图像展示
fig, axes = plt.subplots(1, 5, figsize=(20, 4))  # 设置图像大小为20x4，方便查看
fig.subplots_adjust(wspace=0.2)  # 调整每个子图之间的间距

# 打印每张图片的最大值及其位置，并显示图片
for i, idx in enumerate(random_indices):
    image = data_normalized[idx]  # 获取第idx张图片

    # 获取最大值及其位置
    min_val = torch.min(image)


    # 打印图片的最大值及其位置
    print(f"Image {idx+1}:")
    print(f"Min Value: {min_val}")

    # 在对应的子图中显示图片，并设置颜色条范围为 [0, 1]
    im = axes[i].imshow(image, cmap='viridis', vmin=0, vmax=1)  # 使用 'viridis' 色图，设置颜色条范围为 [0, 1]
    axes[i].set_title(f"Image {idx+1}")
    axes[i].axis('off')  # 关闭坐标轴

    # 添加颜色条
    plt.colorbar(im, ax=axes[i], fraction=0.02, pad=0.04)  # 添加颜色条，确保颜色条在 [0, 1] 范围内

# 显示所有图片
plt.show()

In [ ]:
batch_size = 64
from torch.utils.data import random_split
train_dataset, val_dataset = random_split(
    data_input, 
    [850,151],
    generator=torch.Generator().manual_seed(26) 
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
#test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:

model = VAE_41()
#model.load_state_dict(torch.load('./weight/size41_0130test1.pth')) #网络参数初始化
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LEARNING_RATE = 5e-5
NUM_EPOCHS = 500
#BETA = 0.01 # KL 散度权重
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# if not os.path.exists("./weight"):
#     os.makedirs(".weight")
# ----------------------------------------------------
# 模型的训练函数调用也需要更新，传入 val_loader
# 假设您的 train_vae 函数也返回验证损失历史记录


In [ ]:
import importlib
from Util import train_41utils
importlib.reload(train_41utils)
# reload 后重新取函数（确保指向最新版本）
train_vae_41 = train_41utils.train_vae_41
save_path = "./weight/size41_modulus_0302_1.pth"
trained_model, losses, recon, kl, val_losses, l3_losses = train_vae_41(
    model = model, 
    train_loader = train_loader, 
    val_loader = val_loader, 
    optimizer = optimizer, 
    num_epochs = NUM_EPOCHS, 
    device = device, 
    beta_decay = True, 
    beta = 1, 
    save_path = save_path,
    L3 = True,
    gamma = 10    
    )

In [ ]:

plot_curve(losses, recon, kl, val_losses, l3_losses, log_scale=True)

In [ ]:
#SSIM函数修好了吗好像没有。。。
#ssim_rebuilt, ssim_sample = SSIM(model, test_loader, device)

In [ ]:
import matplotlib.pyplot as plt
def l3_per_sample(x_recon, x, target_pos=(20, 20)):
    x_loc, y_loc = target_pos
    # 适配 (B,1,H,W)；如果你是 (B,H,W) 把 [:,0,...] 改成 [:,...]
    delta = (x_recon[:, 0, x_loc, y_loc] - x[:, 0, x_loc, y_loc])  # shape: (B,)
    xnor = x[:, 0, x_loc, y_loc]  # 避免除0
    return delta / xnor  # (B,)

# 随机挑 batch 里的 K 个样本展示
def show_recon_with_l3(model, loader, device, K=5, target_pos=(20,20), vmin=None, vmax=None, cmap='viridis'):
    model.eval()
    with torch.no_grad():
        batch = next(iter(loader))
        # 适配你的 DataLoader 输出：有的返回 (x, y)，有的只返回 x
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        x = x.to(device)

        # 前向推理（按你 VAE 的 forward 返回来改）
        out = model(x)
        if isinstance(out, (list, tuple)):
            # 常见：x_recon, mu, logvar
            x_recon = out[0]
        else:
            x_recon = out

        l3_vals = l3_per_sample(x_recon, x, target_pos=target_pos).detach().cpu()

        # 从 batch 里挑 K 个索引
        B = x.size(0)
        K = min(K, B)
        idxs = torch.randperm(B)[:K]

        
        for i, idx in enumerate(idxs):
            ori = x[idx, 0].detach().cpu()
            rec = x_recon[idx, 0].detach().cpu()
            l3  = float(l3_vals[idx])

            plt.figure(figsize=(8, 3))
            plt.suptitle(f"sample={int(idx)}   L3@{target_pos}={l3:.6f}")
        
            plt.subplot(1, 2, 1)
            plt.title("Original")
            plt.imshow(ori, cmap=cmap, vmin=vmin, vmax=vmax)
            #plt.scatter([target_pos[1]], [target_pos[0]], s=30)  # 标一下(20,20)
            plt.colorbar()

            plt.subplot(1, 2, 2)
            plt.title("Reconstruction")
            plt.imshow(rec, cmap=cmap, vmin=vmin, vmax=vmax)
            #plt.scatter([target_pos[1]], [target_pos[0]], s=30)
            plt.colorbar()

            plt.tight_layout()
            plt.show()

# 用法：看训练集或验证集随便说明
show_recon_with_l3(trained_model, train_loader, device, K=5, target_pos=(20,20), vmin=0, vmax=1, cmap='viridis')


In [6]:
data_input = np.load(r'..\v41_data\pressure_data.npy')

In [ ]:
model = 

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# =========================
# 0) 你需要准备的变量：
# - data_input: (1000,1,41,41) numpy 或 torch
# - model_path: 权重路径
# - load_model(model_path, device): 你已有的加载函数
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) 取前500条（保持原始顺序）
x500 = data_input[:500]  # (500,1,41,41)

# 2) 转成 torch.Tensor
if isinstance(x500, np.ndarray):
    x500 = torch.from_numpy(x500)
x500 = x500.float()

# 3) 只加载一次模型
model.eval()

# 4) 批量推理拿到 z: (500,8)
bs = 128
Z_list = []
MU_list = []

with torch.no_grad():
    for i in range(0, x500.shape[0], bs):
        xb = x500[i:i+bs].to(device)  # (B,1,41,41)
        # 编码器输出
        mu, logvar = model.encode(xb)
        mu = mu.view(mu.shape[0], -1)   # 保证 (B,8)
        MU_list.append(mu.cpu())
        # 重参数化得到潜在向量
        #z = model.reparameterize(mu, logvar)
        # 有些实现会输出 (B,8,1,1)；这里统一拉平成 (B,8)
        #z = z.view(z.shape[0], -1)
        #Z_list.append(z.detach().cpu())
MU = torch.cat(MU_list, dim=0).numpy()
#Z = torch.cat(Z_list, dim=0).numpy()  # (500,8)
#print("Z shape =", Z.shape)
# idx = np.arange(MU.shape[0])  # 0~499，作为颜色
# plt.figure(figsize=(7, 6))
# sc = plt.scatter(MU[:, 8], MU[:, 9], c=idx, cmap="Blues", s=30)  # Blues: 越大越深
# plt.colorbar(sc, label="Sample index")
# plt.xlabel("Latent dim 1 (mu)")
# plt.ylabel("Latent dim 2 (mu)")
# plt.title("Dim1 vs Dim2 (color darker as index increases)")
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

# 5) 画 8 个维度的趋势（8个子图）
x = np.arange(MU.shape[0])  # 0~499

plt.figure(figsize=(12, 10))
x_min, x_max = x.min(), x.max()
y_min, y_max = MU.min(), MU.max()
for k in range(16):
    ax = plt.subplot(8, 2, k + 1)
    ax.plot(x, MU[:, k])
    ax.set_title(f"Latent dim {k+1}")
    ax.set_xlabel("Sample index (0~499)")
    ax.set_ylabel("Value")
    ax.grid(True, alpha=0.3)
    #ax.set_xlim(x_min, x_max)
    #ax.set_ylim(y_min, y_max)

plt.tight_layout()
plt.show()
'''
# 6)（可选）再给你一个 500×8 的热力图，一眼看整体趋势
plt.figure(figsize=(10, 5))
plt.imshow(Z, aspect='auto')
plt.colorbar(label="Value")
plt.xlabel("Latent dimension (0~7)")
plt.ylabel("Sample index (0~499)")
plt.title("Latent vectors heatmap (500x8)")
plt.tight_layout()
plt.show()
'''


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3) 只加载一次模型
model.eval()

# 4) 批量推理拿到 z: (500,8)
bs = 50
Z_list = []
MU_list = []

with torch.no_grad():
    for i in range(0, data_input.shape[0], bs):
        xb = data_input[i:i+bs].to(device)  # (B,1,41,41)
        # 编码器输出
        mu, logvar = model.encode(xb)
        mu = mu.view(mu.shape[0], -1)   # 保证 (B,8)
        MU_list.append(mu.cpu())
        # 重参数化得到潜在向量
        #z = model.reparameterize(mu, logvar)
        # 有些实现会输出 (B,8,1,1)；这里统一拉平成 (B,8)
        #z = z.view(z.shape[0], -1)
        #Z_list.append(z.detach().cpu())
MU = torch.cat(MU_list, dim=0).numpy()
#Z = torch.cat(Z_list, dim=0).numpy()  # (500,8)
#print("Z shape =", Z.shape)


In [ ]:
x = np.arange(MU.shape[0])  # 0~499

plt.figure(figsize=(12, 10))
for k in range(16):
    ax = plt.subplot(8, 2, k + 1)
    ax.plot(x, MU[:, k])
    ax.set_title(f"Latent dim {k+1}")
    ax.set_xlabel("Sample index (0~499)")
    ax.set_ylabel("Value")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
sc = plt.scatter(MU[:, 1], MU[:, 2], c=idx, cmap="Blues", s=30)  # Blues: 越大越深
plt.colorbar(sc, label="Sample index")
plt.xlabel("Latent dim 1 (mu)")
plt.ylabel("Latent dim 2 (mu)")
plt.title("Dim1 vs Dim2 (color darker as index increases)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


# 创建 PCA 对象，选择保留的主成分数目，可以选择2个或者更多主成分
pca = PCA(n_components=2)  # 降维到 2 维

# 对数据进行主成分分析
reduced_data = pca.fit_transform(MU)

# 输出主成分
print("主成分矩阵：")
print(pca.components_)  # 每个主成分的方向
print("解释方差比例：")
print(pca.explained_variance_ratio_)  # 每个主成分解释的方差比例

# 绘制降维后的数据
idx = np.arange(MU.shape[0])
plt.scatter(reduced_data[:, 0], reduced_data[:, 1], c=idx, cmap="Blues", s=30)
plt.title('PCA of 16-Dimensional Hidden Vectors')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()




In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms



# 假设模型的编码器和解码器分别是：
encoder = model.encode
decoder = model.decode

# 假设你有一个原始样本 x，来生成潜在空间中的点
x1 = data_input[500]
x2 = data_input[550]
x1 = x1.unsqueeze(0).to(device)
x2 = x2.unsqueeze(0).to(device)

z1, _ = encoder(x1)  # 编码器得到潜在空间中的 z1
z2, _ = encoder(x2)  # 假设我们也可以从另一张图片获得 z2


# 线性插值生成潜在空间中的中间点
lambdas = np.linspace(0, 1, 10)  # 生成 10 个插值点
interpolated_z = [(1 - lam) * z1 + lam * z2 for lam in lambdas]

# 使用解码器生成对应的图像
generated_images = []
for z in interpolated_z:
    z = z.unsqueeze(0)  # 增加 batch 维度
    with torch.no_grad():
        generated_image = decoder(z)  # 解码器生成图像
    generated_images.append(generated_image.squeeze(0).cpu())

# 可视化生成的图像
fig, axes = plt.subplots(1, len(generated_images), figsize=(15, 5))
for i, image in enumerate(generated_images):
    axes[i].imshow(image.permute(1, 2, 0))  # 转换为 HWC 格式
    axes[i].axis('off')
plt.show()

# 计算相邻样本之间的 L2 距离（或其他相似度度量）
l2_distances = []
for i in range(1, len(generated_images)):
    prev_image = generated_images[i - 1].view(-1)
    curr_image = generated_images[i].view(-1)

# 输出 L2 距离
print("L2 Distances between consecutive images:", l2_distances)

# 可视化 L2 距离变化
plt.plot(lambdas[1:], l2_distances, marker='o')
plt.title("L2 Distance between consecutive generated samples")
plt.xlabel("Lambda (Interpolation Parameter)")
plt.ylabel("L2 Distance")
plt.grid(True)
plt.show()


In [ ]:
model = VAE_41()
model.load_state_dict(torch.load('./weight/size41_modulus_0226_2.pth')) #网络参数初始化


In [ ]:
#两样本插值
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = model.encode
decoder = model.decode

x1 = data_input[350]
x2 = data_input[150]
x1 = x1.unsqueeze(0).to(device)
x2 = x2.unsqueeze(0).to(device)

z1, _ = encoder(x1)  # 编码器得到潜在空间中的 z1
z2, _ = encoder(x2)  # 假设我们也可以从另一张图片获得 z2
#print(z1)
#print(z2)
lambdas = torch.linspace(0, 1, 11, device=z1.device, dtype=z1.dtype)
vector = []
for d in range(16):
    z_all = z1.repeat(11, 1)
    z_all[:, d] = (1 - lambdas) * z1[0, d] + lambdas * z2[0, d]
    #print(z_all[:,d])
    vector.append(z_all)

In [ ]:
import matplotlib.pyplot as plt
model.eval()
fig, axes = plt.subplots(4, 4, figsize=(16, 12), sharex=True, sharey=True)
axes = axes.ravel()
for dim_idx in range(16):
    ax = axes[dim_idx]
    
    with torch.no_grad():
        x_rec = decoder(vector[dim_idx])          # 常见：decoder直接接收 (B, latent_dim)
        mins = x_rec.view(x_rec.size(0), -1).min(dim=1).values  # shape: (10,)
    
    min_list = mins.detach().cpu().tolist()

    ax.plot(range(1, len(min_list)+1), min_list, marker='o')
    ax.set_title(f"dim {dim_idx}")
    ax.grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", bbox_to_anchor=(1.02, 1), title="images")

fig.supxlabel("chazhijiange")
fig.supylabel("min_value")
plt.tight_layout()
plt.show()